In [1]:
import os
import time
import json
import shutil
import argparse
import numpy as np

import functools
import operator

import pickle, gzip

import uproot

In [3]:
def combine_vineReduce_channels(pkl): 
    vineReduce_dict = load_pickle(pkl_file)
    first_ch = next(iter(vineReduce_dict))
    variables = vineReduce_dict[first_ch].keys()
    
    hists = {}
    for var in variables: 
        h_list = [vineReduce_dict[ch][var] for ch in vineReduce_dict]
        hists[var] = functools.reduce(operator.add, h_list)

    return hists

def flatten_vineReduce_hists(pkl, outname = 'flat_hists'):
    # Flattens the histogram structure from vineReduce and saves as a new pickle. 
    # Returns the name of the new pickle file 
    
    flattend_hists = combine_vineReduce_channels(pkl)
    tmp_pkl = f"{outname}.pkl.gz"
    with gzip.open(tmp_pkl, "wb") as f:
        pickle.dump(flattend_hists, f)
    
    return tmp_pkl

def load_pickle(fname):
    return pickle.load(gzip.open(fname))

In [4]:
file_path = "test_200424_full/TESTING-ttbar-ee_2b_2j_mllbb.root" 
with uproot.open(file_path) as f:
    print(f"Inspecting file: {file_path}")

    # Get all histogram names (keys)
    all_keys = f.keys(cycle=False)

    # for key in all_keys:
    for key in ['tt_sm_elecIDUp', 'tt_sm_elecIDDown']:
        print(f"{key}")
        print(f[key].values())

Inspecting file: test_200424_full/TESTING-ttbar-ee_2b_2j_mllbb.root
tt_sm_elecIDUp
[4.55211006e+01 2.84796939e+02 1.02475672e+03 1.61534667e+03
 1.99568009e+03 2.00786482e+03 1.74225487e+03 1.45563657e+03
 1.05047310e+03 6.93460328e+02 4.09928640e+02 2.24377960e+02
 1.46109062e+02 5.31342180e+01 9.64899733e+00 6.09458530e-01]
tt_sm_elecIDDown
[4.28226925e+01 2.67450941e+02 9.60671992e+02 1.51113986e+03
 1.86362084e+03 1.87216787e+03 1.62214263e+03 1.35346229e+03
 9.75395832e+02 6.42643295e+02 3.79251822e+02 2.07108375e+02
 1.34360865e+02 4.85075189e+01 8.77034589e+00 5.48525916e-01]


In [5]:
pkl_file = "../analysis/SR_ALL_260424.pkl.gz"
prepared_pkl = flatten_vineReduce_hists(pkl_file)

In [6]:
hist_dict = load_pickle(prepared_pkl)

In [7]:
channels = list(hist_dict['mllbb'].axes['channel'])
all_procs = list(hist_dict['mllbb'].axes['process'])
systs = list(hist_dict['mllbb'].axes['systematic'])
systs.remove('sumw2')
data_procs = [x for x in all_procs if 'data' in x]
MC_procs = [x for x in all_procs if x not in data_procs]

In [11]:
tt_procs = [x for x in MC_procs if 'TT01j2l' in x]
tW_procs = [x for x in MC_procs if 'TW' in x]
DY_procs = [x for x in MC_procs if 'DY' in x]
Others_procs = [x for x in MC_procs if ((x not in tt_procs) and (x not in tW_procs) and (x not in DY_procs))]

print(DY_procs)

['DY10to50_centralUL16APV', 'DY50_centralUL16APV', 'DY10to50_centralUL16', 'DY50_centralUL16', 'DY10to50_centralUL17', 'DY50_centralUL17', 'DY10to50_centralUL18', 'DY50_centralUL18']


In [9]:
# base_procs = ['tt', 'tW', 'DY', 'Others']

grouped_procs = {
    'tt':tt_procs,
    'tW':tW_procs,
    'DY':DY_procs,
    'Others': Others_procs
}

In [15]:
# Checks that the bin values in the histogram for combine matches what I have from the orignal histogram file
# Doesn't work for systematics that are specific to the year, those are handled separately below. 
for ch in channels: 
    print(f"channel: {ch}")
    root_file = os.path.join("test_200424_full", f"TESTING-ttbar-{ch}_mllbb.root")
    with uproot.open(root_file) as f:
        print(f"Inspecting file: {root_file}")
        all_keys = f.keys(cycle=False)
        # print(all_keys)
        for p in grouped_procs.keys(): 
            print(f"  proc: {p}")
            correct_systs = []
            wrong_systs = []
            proc_list = grouped_procs[p]
            for s in systs:
                if 'fact' in s or 'renorm' in s: 
                    continue
                if ('2016' in s) or ('2017' in s) or ('2018' in s): 
                    continue
                if s == 'nominal': 
                    key_name = f"{p}_sm"
                else: 
                    key_name = f"{p}_sm_{s}"
                root_values = f[key_name].values()
                orig_hist = hist_dict['mllbb'][{'channel':ch, 'systematic':s, 'process':proc_list}][{'process':sum}].as_hist({})
                orig_values = orig_hist.values()
                matches = np.allclose(root_values[:15], orig_values)
                if matches: 
                    correct_systs.append(s)
                if not matches: 
                    wrong_systs.append(s)
                    
            if p == "DY":
                print(f"orig hist vals: \n{hist_dict['mllbb'][{'channel':ch, 'systematic':'elecIDUp', 'process':proc_list}][{'process':sum}].as_hist({}).values()}")
                print(f"combine hist: \n{f[f'{p}_sm_{s}'].values()}")
            # print(f"\tsysts that match: {correct_systs}")
            print(f"\tsysts that DO NOT match: {wrong_systs} \n\n")

channel: ee_2b_2j
Inspecting file: test_200424_full/TESTING-ttbar-ee_2b_2j_mllbb.root
  proc: tt
	systs that DO NOT match: [] 


  proc: tW
	systs that DO NOT match: [] 


  proc: DY
orig hist vals: 
[ 0.          0.67244231  4.02609305 16.8093138  13.33986758  4.12158375
 -9.35656096 10.15658761 -0.10214728  9.47019097  4.13380172  0.80515182
  4.52525211  0.          1.50798858]
combine hist: 
[ 0.          0.64693499  3.87967386 16.25849271 12.82871173  3.98110542
  0.          9.82071399  0.          9.06251214  3.93360683  0.76797476
  4.34497272  0.          1.37568357  0.        ]
	systs that DO NOT match: ['nominal', 'elecIDUp', 'elecIDDown', 'muonIDUp', 'muonIDDown', 'muonISOUp', 'muonISODown', 'trigSFUp', 'trigSFDown', 'L1prefireUp', 'L1prefireDown', 'PUUp', 'PUDown', 'jetPuIDUp', 'jetPuIDDown', 'btagSFbc_correlatedUp', 'btagSFbc_correlatedDown', 'btagSFlight_correlatedUp', 'btagSFlight_correlatedDown', 'FSRUp', 'FSRDown', 'ISRUp', 'ISRDown', 'hdampUp', 'hdampDown', 'JES_Abso

In [8]:
file_path = "test_200427_addPDF/TESTING-ttbar-ee_2b_2j_mllbb.root" 
with uproot.open(file_path) as f:
    print(f"Inspecting file: {file_path}")

    # Get all histogram names (keys)
    all_keys = f.keys(cycle=False)
    # print(all_keys)
    # for key in all_keys:
    for key in ['tt_sm', 'tt_sm_PDFUp', 'tt_sm_PDFDown']:
        print(f"{key}")
        print(f[key].values())

Inspecting file: test_200427_addPDF/TESTING-ttbar-ee_2b_2j_mllbb.root
tt_sm
[4.41574261e+01 2.76027316e+02 9.92345421e+02 1.56261942e+03
 1.92883648e+03 1.93915750e+03 1.68141932e+03 1.40387448e+03
 1.01243063e+03 6.67700966e+02 3.94374470e+02 2.15618932e+02
 1.40146690e+02 5.07837333e+01 9.20215738e+00 5.78395254e-01]
tt_sm_PDFUp
[  44.65017718  278.92345087 1002.43858856 1577.70977611 1946.9142935
 1956.64835506 1696.61833843 1416.99302014 1021.98207586  674.78004276
  398.91812511  218.50573666  141.78141697   51.4175887     9.34695818]
tt_sm_PDFDown
[  43.66467499  273.13118134  982.2522541  1547.52906759 1910.75866385
 1921.66665027 1666.22030072 1390.75593664 1002.87918102  660.6218889
  389.83081539  212.73212755  138.5119636    50.14987788    9.05735659]


In [2]:
file_path = "test_200427_addPDF/TESTING-ttbar-ee_2b_2j_mllbb.root" 
with uproot.open(file_path) as f:
    print(f"Inspecting file: {file_path}")

    # Get all histogram names (keys)
    all_keys = f.keys(cycle=False)
    # print(all_keys)
    # for key in all_keys:
    for key in ['tt_sm', 'tt_sm_PDFUp', 'tt_sm_PDFDown']:
        print(f"{key}")
        print(f[key].values())

Inspecting file: test_200427_addPDF/TESTING-ttbar-ee_2b_2j_mllbb.root
tt_sm
[4.41574261e+01 2.76027316e+02 9.92345421e+02 1.56261942e+03
 1.92883648e+03 1.93915750e+03 1.68141932e+03 1.40387448e+03
 1.01243063e+03 6.67700966e+02 3.94374470e+02 2.15618932e+02
 1.40146690e+02 5.07837333e+01 9.20215738e+00 5.78395254e-01]
tt_sm_PDFUp
[4.46501772e+01 2.78923451e+02 1.00243859e+03 1.57770978e+03
 1.94691429e+03 1.95664836e+03 1.69661834e+03 1.41699302e+03
 1.02198208e+03 6.74780043e+02 3.98918125e+02 2.18505737e+02
 1.41781417e+02 5.14175887e+01 9.34695818e+00 5.87496609e-01]
tt_sm_PDFDown
[4.36646750e+01 2.73131181e+02 9.82252254e+02 1.54752907e+03
 1.91075866e+03 1.92166665e+03 1.66622030e+03 1.39075594e+03
 1.00287918e+03 6.60621889e+02 3.89830815e+02 2.12732128e+02
 1.38511964e+02 5.01498779e+01 9.05735659e+00 5.69293900e-01]
